In [1]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline 

words = open("data-raw/names.txt").read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [2]:
# build the vocabulary 
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {stoi[ch]:ch for ch in stoi}

In [3]:
block_size = 5

def build_dataset(words, block_size):
  X,Y = [],[]
  for w in words:
    #print(w)
    context = [0] * block_size
    for ch in (list(w) + ['.']):
      ix = stoi[ch]
      X.append(list(context))
      Y.append(ix)
      #print(''.join(itos[i] for i in context), ' --> ', itos[ix])
      # context.pop(0)
      # context.append(ix)
      context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

import random 
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2  =int(0.9 * len(words))
         
Xtr, Ytr = build_dataset(words[:n1], block_size)
Xdev, Ydev = build_dataset(words[n1:n2], block_size)
Xtst, Ytst = build_dataset(words[n2:], block_size)


In [4]:
num_tokens = len(stoi)

# hyperparameters 
num_embeddings = 10 
num_nodes = 300

# Initialize
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((num_tokens, num_embeddings), generator=g, requires_grad=True)
W1 = torch.randn((block_size * num_embeddings, num_nodes), generator=g, requires_grad=True)
b1 = torch.randn(num_nodes, generator=g, requires_grad=True)
W2 = torch.randn((num_nodes, num_tokens), generator=g, requires_grad=True)
b2 = torch.randn(num_tokens, generator=g, requires_grad=True)

parameters = [C, W1, b1, W2, b2]

sum(p.nelement() for p in parameters)

23697

In [5]:
def forward_pass(X, Y, batch_size):
  # minibatch 
  if batch_size != -1:
    ix = torch.randint(0, X.shape[0], (batch_size,), generator=g)
  else:
    ix = torch.arange(X.shape[0])

  #forward pass 
  h = torch.tanh(C[X[ix]].view((-1 , num_embeddings*block_size )) @ W1 + b1)
  logits = (h @ W2 + b2)
  loss = F.cross_entropy(logits, Y[ix])
  return loss

def backward_pass(loss, parameters, eta):
  for p in parameters:
    p.grad  = None 
  loss.backward()
  # update 
  for p in parameters:
    p.data += -eta.item() * p.grad

#
def train(X, Y, iters):        
  eta = 0.1
  for k in range(iters):
    loss = forward_pass(X, Y, 32)

    # backward pass
    #backward_pass(parameters, eta[k])
    for p in parameters:
      p.grad  = None 
    loss.backward()

    if k > iters/2: 
      eta = 0.01
    for p in parameters:
      p.data += -eta * p.grad
  #print(loss.item())

train(Xtr, Ytr, 200000)

In [6]:
print(forward_pass(Xtr, Ytr, -1).item())
print(forward_pass(Xdev, Ydev, -1).item())
print(forward_pass(Xtst, Ytst, -1).item())

2.17876935005188
2.2032344341278076
2.1989729404449463
